# Phase 7 - Prior case retrieval with Arch 3 - Llama-style decoder

Starts from the Phase-1 model (12-layer Llama-style decoder (100.1M): RMSNorm, RoPE, grouped-query attention, SwiGLU, byte-level BPE.) and, for every query judgment, ranks the cases in its pool and returns
the **top 7** most likely to be cited. Metric: micro-F1@7.

**What the data asks for.** Train has 827 queries against a pool of 4,320 cases (dev 118 / 1,023, test 237 / 1,727);
each query cites ~6-8 of them. Judgments are long and names, organisations and citations are masked (`<NAME>`,
`<ORG>`, `<CITATION>`), so the signal is the legal content itself. Every query also sits in its own pool - it is never
allowed to retrieve itself.

**Approach: hybrid dense + lexical retrieval.**
* **Dense bi-encoder** (Karpukhin et al. 2020, DPR): the decoder with its causal mask lifted reads three evenly spaced 512-token windows of a case
  (opening, middle, end - MaxP/FirstP-style passage evidence, Dai & Callan 2019); the windows are mean-pooled and
  projected to one normalised 256-d vector. Queries and candidates share the encoder.
* **Contrastive training**: InfoNCE with in-batch negatives plus **BM25 hard negatives** (Karpukhin et al. 2020;
  Xiong et al. 2021, ANCE); candidates relevant to another query in the batch are masked rather than punished.
* **BM25** (Robertson & Zaragoza 2009), written from scratch, on rare query terms only - this alone is a strong
  baseline for legal PCR (Joshi et al. 2023, IL-PCR / U-CREAT).
* **Fusion**: `a * z(dense) + (1 - a) * z(BM25)`, with `a` tuned on dev (convex score fusion, Bruch et al. 2023).

**Experiments** (each is its own submission; all from the Phase-7 list)

| Experiment | What trains |
|---|---|
| `full` | every weight - strong reference and the distillation teacher |
| `lorma` | **LoRMA** (compulsory): `(alpha/r BA + I) W x` on the attention projections (Bihany, Patel & Modi 2025) |
| `partial` | only the top 4 layers (+ final norm and projection) |
| `ia3` | IA3: learned rescaling of keys/values and feed-forward activations (Liu et al. 2022) |
| `prompt` | prompt tuning: 16 learned virtual tokens in front of every window (Lester et al. 2021) |
| `kd` | distillation (Hinton et al. 2015): a half-depth student learns the fine-tuned `full` model's query-candidate similarity distribution and its embeddings |

Every experiment trains for a fixed **time budget** (`TRAIN_MINUTES`); the checkpoint with the best dev micro-F1@7 is kept.

**Kaggle settings:** GPU T4 x2, Internet on, secret `HF` holding your Hugging Face token. Then Run All.

## 1. Setup

In [1]:
import os, sys, json, glob, shutil, time, gc
import numpy as np
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/phase7_arch3"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GPUs: ['Tesla T4', 'Tesla T4']


## 2. Configuration
Everything you might want to change lives here.

In [2]:
%%writefile /kaggle/working/phase7_arch3/config.py
"""Every knob of this Phase-7 (prior case retrieval) notebook."""
import os
import json

ARCH = "arch3"
WORK_DIR = f"/kaggle/working/phase7_{ARCH}"
DATA_DIR = os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "phase1")        # Phase-1 files downloaded from Hugging Face land here
OUT_DIR = os.path.join(WORK_DIR, "runs")             # one folder per experiment

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SPLIT_FOLDERS = {"train": "Train", "dev": "Dev", "test": "Test"}
RAW = {s: (os.path.join(DATA_DIR, "Phase_7_PCR", f, f"{s}.json"), os.path.join(DATA_DIR, "Phase_7_PCR", f, f"{s}_candidates.json"))
       for s, f in SPLIT_FOLDERS.items()}

# ------------------------------------------------------------------ the Phase-1 model
# Looked up in this order: an attached Kaggle dataset whose path mentions "arch3", then these HF repos.
PHASE1_REPOS = ["D-240357/legal-arch3-decoder"]
WEIGHT_FILES = ["model_final.pt", "model_best.pt"]
TOKENIZER_FILES = ["tokenizer.json", "data/tokenizer.json"]
TOKENIZER_REPOS = PHASE1_REPOS
KAGGLE_HINT = "arch3"

PCR_REPO = "D-240357/legal-pcr-arch3"           # fine-tuned checkpoints, results and submission zips go here

# ------------------------------------------------------------------ experiments
# Each one is a separate submission. "full" is the strong reference and the distillation teacher;
# "kd" waits for it to finish.
METHODS = ["full", "lorma", "partial", "ia3", "prompt", "kd"]
KD_TEACHER = "full"
TRAIN_MINUTES = 30             # wall-clock training budget per experiment (the LR schedule ends exactly on it)
MAX_EPOCHS = 12

# ------------------------------------------------------------------ how a case is read
CHUNK_TOKENS = 512
MAX_CHUNKS = 3                 # three evenly spaced 512-token windows (opening, middle, end) are mean-pooled
TRAIN_CHUNKS = 2               # random subset of them read per case while training (saves memory)
CHAR_CAP = 300_000             # text beyond this is not even tokenized
BIDIRECTIONAL = True            # lift the causal mask while fine-tuning (Dukic & Snajder 2024; LLM2Vec)
EMB_DIM = 256                  # size of the case embedding

# ------------------------------------------------------------------ BM25 (also gives the hard negatives)
BM25_MIN_IDF = 1.0             # the query keeps only words that are rare enough to say something ...
BM25_MAX_TERMS = 300           # ... and at most its 300 most telling ones (tf x idf)
HARD_NEG_POOL = 30             # hard negatives are drawn from the top-30 BM25 candidates that are not relevant
FUSION_GRID = [round(0.05 * i, 2) for i in range(21)]   # final score = a z(dense) + (1 - a) z(BM25), a tuned on dev

# ------------------------------------------------------------------ optimisation
TEMPERATURE = 0.05             # InfoNCE temperature
MICRO_QUERIES = 4              # queries per forward pass: each brings one positive and one hard negative
GRAD_ACCUM = 2
EVAL_BATCH = 16
WARMUP_STEPS = 30
MIN_LR_RATIO = 0.1
WEIGHT_DECAY = 0.01
HEAD_LR = 1e-3
FULL_LR = 2e-5
PARTIAL_LR = 5e-5
PARTIAL_TOP_LAYERS = 4
ADAPTER_LR = 5e-4
ADAPTER_RANK = 8
LORMA_ALPHA = 8
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["qkv", "proj"]       # LoRMA: attention projections
IA3_OUT_TARGETS = ["qkv"]       # key/value projections
IA3_IN_TARGETS = ["down"]         # feed-forward down projections
PROMPT_LEN = 16
PROMPT_LR = 1e-2

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5                 # weight of the contrastive loss; the rest goes to matching the teacher's similarities
KD_HIDDEN_WEIGHT = 0.5         # also align the student's case embeddings with the teacher's (cosine)
STUDENT_LAYERS = [0, 2, 4, 6, 8, 10]

# ------------------------------------------------------------------ evaluation
EVAL_EVERY_STEPS = 100         # full dev ranking (118 queries against 1,023 candidates); the best checkpoint is kept
TRAIN_QUERY_LIMIT = None       # set to e.g. 40 (and TRAIN_MINUTES = 2) for a quick smoke test
PREP_WORKERS = max(1, os.cpu_count() or 2)

SEED = 13

_override = os.environ.get("P7_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

Writing /kaggle/working/phase7_arch3/config.py


## 3. Code
The Phase-1 tokenizer and model (unchanged, so the trained weights load), plus the retrieval pieces.

In [3]:
%%writefile /kaggle/working/phase7_arch3/tokenizer.py
"""
Byte-level BPE tokenizer, written from scratch (stdlib only for the tokenizer logic).

How it works, in short:
  1. A regex chops text into "pre-tokens" (a word with its leading space, a run of digits,
     a run of punctuation, ...). Merges never cross these boundaries.
  2. Every pre-token is turned into raw UTF-8 bytes (ids 0..255), so *any* text can be
     encoded - there is no <unk> and no lowercasing.
  3. Training repeatedly merges the most frequent adjacent id pair into a new id.
     We train on the table of *unique* pre-tokens with their counts, which is what keeps
     it fast in pure Python.
  4. Encoding replays the learned merges (lowest rank first) per pre-token, and caches the
     result, so repeated words cost a single dict lookup.
"""
import re
import json
import heapq
from collections import Counter, defaultdict

# Close to the GPT-2 splitter, but built on plain `re` (no \p{L}), so it needs no extra package.
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+",
    re.UNICODE,
)

EOS_TOKEN = "<|endoftext|>"
PAD_TOKEN = "<|pad|>"


class ByteBPE:
    def __init__(self, merges=None, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = [tuple(m) for m in (merges or [])]      # merge i creates id 256 + i
        self.eos_id = vocab_size - 2
        self.pad_id = vocab_size - 1
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------- internals
    def _rebuild_tables(self):
        self.ranks = {pair: (i, 256 + i) for i, pair in enumerate(self.merges)}
        vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            vocab.append(vocab[a] + vocab[b])
        self.vocab = vocab
        self._cache.clear()

    # -------------------------------------------------------------- training
    @classmethod
    def train(cls, texts, vocab_size=32000, log_every=2000):
        """Learn merges from an iterable of strings."""
        n_merges = vocab_size - 256 - 2          # leave room for the two special tokens
        word_counts = Counter()
        for text in texts:
            word_counts.update(PRETOKEN_RE.findall(text))
        print(f"  unique pre-tokens: {len(word_counts):,}")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_count = defaultdict(int)
        where = defaultdict(set)                 # pair -> indices of words that (may) contain it
        for wi, w in enumerate(words):
            f = freqs[wi]
            for pair in zip(w, w[1:]):
                pair_count[pair] += f
                where[pair].add(wi)

        # Max-heap with lazy invalidation: stale entries are fixed up when they surface.
        heap = [(-c, p) for p, c in pair_count.items()]
        heapq.heapify(heap)

        merges = []
        vocab = [bytes([i]) for i in range(256)]
        for m in range(n_merges):
            best = None
            while heap:
                neg, pair = heapq.heappop(heap)
                real = pair_count.get(pair, 0)
                if real <= 0:
                    continue
                if real != -neg:
                    heapq.heappush(heap, (-real, pair))
                    continue
                best = pair
                break
            if best is None:
                print(f"  ran out of pairs after {m} merges")
                break

            new_id = 256 + m
            merges.append(best)
            a, b = best
            vocab.append(vocab[a] + vocab[b])
            touched = {}
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                # find the occurrences in this word (left to right, non-overlapping)
                if len(w) < 2:
                    continue
                i, out, changed = 0, [], False
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                        changed = True
                    else:
                        out.append(w[i])
                        i += 1
                if not changed:
                    continue                      # stale entry in `where`
                for pair in zip(w, w[1:]):        # take the old word's pairs away ...
                    pair_count[pair] -= f
                for pair in zip(out, out[1:]):    # ... and add the new word's pairs
                    pair_count[pair] += f
                    where[pair].add(wi)
                    touched[pair] = pair_count[pair]
                words[wi] = out
            for pair, c in touched.items():
                if c > 0:
                    heapq.heappush(heap, (-c, pair))
            pair_count.pop(best, None)

            if (m + 1) % log_every == 0:
                print(f"  merge {m + 1:,}/{n_merges:,}  last = {vocab[-1]!r}")

        return cls(merges, vocab_size)

    # -------------------------------------------------------------- encoding
    def _encode_word(self, word):
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) > 1:
            best_rank, best_pair, best_new = None, None, None
            for pair in zip(ids, ids[1:]):
                hit = ranks.get(pair)
                if hit is not None and (best_rank is None or hit[0] < best_rank):
                    best_rank, best_pair, best_new = hit[0], pair, hit[1]
            if best_pair is None:
                break
            a, b = best_pair
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(best_new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out
        return ids

    def encode(self, text):
        cache = self._cache
        if len(cache) > 1_000_000:               # keep RAM bounded on huge corpora
            cache.clear()
        out = []
        for word in PRETOKEN_RE.findall(text):
            ids = cache.get(word)
            if ids is None:
                ids = cache[word] = self._encode_word(word)
            out.extend(ids)
        return out

    def decode(self, ids):
        specials = (self.eos_id, self.pad_id)
        data = b"".join(self.vocab[i] for i in ids if i < len(self.vocab) and i not in specials)
        return data.decode("utf-8", errors="replace")

    # ----------------------------------------------------------- persistence
    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges,
                       "eos": EOS_TOKEN, "pad": PAD_TOKEN}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            blob = json.load(f)
        return cls(blob["merges"], blob["vocab_size"])


# ---------------------------------------------------------------------------------------------------
# Phase 2 addition: encode with character offsets, so predicted tokens can be mapped back to exact spans.
# ---------------------------------------------------------------------------------------------------
# Same splitter as above, except a run of punctuation is cut into single characters. Gold entities often end
# in the middle of runs like "Ors.[1]" or "Cr.P.C.,"; with this split 99.3% of gold spans start and end on a
# token boundary (98.1% without it). Every piece is still encoded with the Phase-1 merges.
NER_PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]|\s+(?!\S)|\s+",
    re.UNICODE,
)


class NerTokenizer(ByteBPE):
    split_punct = True

    def encode_with_offsets(self, text):
        """Returns ids and, per token, the [start, end) of its non-whitespace characters in `text`."""
        splitter = NER_PRETOKEN_RE if self.split_punct else PRETOKEN_RE
        ids, starts, ends = [], [], []
        for m in splitter.finditer(text):
            word, base = m.group(), m.start()
            toks = self._cache.get(word)
            if toks is None:
                toks = self._cache[word] = self._encode_word(word)
            # byte position -> character position inside this piece (pieces can contain multi-byte characters)
            char_at_byte = []
            for ci, ch in enumerate(word):
                char_at_byte += [ci] * len(ch.encode("utf-8"))
            b = 0
            for t in toks:
                n = len(self.vocab[t])
                cs, ce = char_at_byte[b], char_at_byte[b + n - 1] + 1
                b += n
                piece = word[cs:ce]
                s = base + cs + (len(piece) - len(piece.lstrip()))
                e = base + ce - (len(piece) - len(piece.rstrip()))
                if s >= e:
                    s = e = base + ce
                ids.append(t)
                starts.append(s)
                ends.append(e)
        return ids, starts, ends

Writing /kaggle/working/phase7_arch3/tokenizer.py


In [4]:
%%writefile /kaggle/working/phase7_arch3/phase1_model.py
"""
Phase-1 model code, unchanged except for the small switches marked "Phase 2", so the saved weights load as-is.
"""
"""
Arch 3: a compact Llama-style decoder, written from scratch in plain PyTorch ops.

  * RMSNorm (pre-norm), no biases anywhere
  * Rotary position embeddings (RoPE) instead of a learned position table
  * Grouped-query attention: fewer KV heads than query heads -> smaller KV cache, fewer params
  * Attention goes through F.scaled_dot_product_attention, so the T x T score matrix is never
    materialised (fused flash / memory-efficient kernel is picked automatically)
  * SwiGLU feed-forward
  * Tied input/output embeddings
  * KV-cache generation, batched, with left-padded prompts
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 32000
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: int = 4
    d_model: int = 768
    d_ff: int = 2048
    max_seq_len: int = 2048
    rope_base: float = 10000.0
    norm_eps: float = 1e-5
    z_loss: float = 1e-4


class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # statistics in fp32 even when the surrounding math is fp16
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32 * self.weight).to(x.dtype)


def build_rope_cache(head_dim, max_len, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)       # (T, head_dim/2)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    # x: (B, heads, T, head_dim); rotate the two halves of every head vector
    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.n_kv = cfg.n_kv_head
        self.head_dim = cfg.d_model // cfg.n_head
        self.group = cfg.n_head // cfg.n_kv_head
        q_dim = self.n_head * self.head_dim
        kv_dim = self.n_kv * self.head_dim
        self.qkv = nn.Linear(cfg.d_model, q_dim + 2 * kv_dim, bias=False)   # one fused matmul
        self.proj = nn.Linear(q_dim, cfg.d_model, bias=False)
        self.q_dim, self.kv_dim = q_dim, kv_dim
        self.causal = True                         # Phase 2 turns this off to let tokens look right

    def forward(self, x, cos, sin, past=None, key_mask=None):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split([self.q_dim, self.kv_dim, self.kv_dim], dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv, self.head_dim).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past is not None:                       # decoding: stash un-expanded K/V for reuse
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        present = (k, v)

        if self.group > 1:                         # share each KV head across `group` query heads
            k = k.repeat_interleave(self.group, dim=1)
            v = v.repeat_interleave(self.group, dim=1)

        if key_mask is None:
            # training path: plain causal attention, fastest kernel
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)

        out = out.transpose(1, 2).reshape(B, T, self.q_dim)
        return self.proj(out), present


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_up = nn.Linear(cfg.d_model, 2 * cfg.d_ff, bias=False)     # fused gate + up
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        gate, up = self.gate_up(x).chunk(2, dim=-1)
        return self.down(F.silu(gate) * up)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg)

    def forward(self, x, cos, sin, past=None, key_mask=None):
        a, present = self.attn(self.attn_norm(x), cos, sin, past, key_mask)
        x = x + a
        x = x + self.ffn(self.ffn_norm(x))
        return x, present


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)

        cos, sin = build_rope_cache(cfg.d_model // cfg.n_head, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # GPT-2 trick: shrink the layers that write into the residual stream so the
        # variance does not grow with depth.
        scale = 0.02 / math.sqrt(2 * cfg.n_layer)
        for blk in self.blocks:
            nn.init.normal_(blk.attn.proj.weight, std=scale)
            nn.init.normal_(blk.ffn.down.weight, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())        # tied weights are counted once

    def hidden_states(self, idx):
        """Final-layer hidden states (B, T, d_model) - what a Phase 2-7 task head sits on top of."""
        B, T = idx.shape
        cos = self.rope_cos[:T]
        sin = self.rope_sin[:T]
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x, _ = blk(x, cos, sin)
        return self.final_norm(x)

    def forward(self, idx, targets=None, caches=None, start_pos=0, key_mask=None, last_only=False):
        """
        Training:   model(idx, targets)            -> (ce_loss, z_loss)
        Inference:  model(idx, caches=..., ...)    -> (logits, new_caches)
        """
        B, T = idx.shape
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]

        x = self.tok_emb(idx)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            past = caches[i] if caches is not None else None
            x, present = blk(x, cos, sin, past, key_mask)
            new_caches.append(present)
        x = self.final_norm(x)
        if last_only:
            x = x[:, -1:, :]

        logits = F.linear(x, self.tok_emb.weight)               # tied head

        if targets is None:
            return logits, new_caches

        logits = logits.float()
        ce = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        z = torch.logsumexp(logits, dim=-1).pow(2).mean()
        return ce, z


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
def _sample(logits, temperature, top_k, top_p):
    """logits: (B, V) fp32 -> next token ids (B,)"""
    logits = logits / max(temperature, 1e-5)
    k = min(top_k, logits.size(-1)) if top_k > 0 else logits.size(-1)
    vals, idx = torch.topk(logits, k, dim=-1)                   # already sorted, descending
    probs = F.softmax(vals, dim=-1)
    if top_p < 1.0:
        before = probs.cumsum(-1) - probs                       # mass strictly before each token
        vals = vals.masked_fill(before >= top_p, float("-inf"))
        probs = F.softmax(vals, dim=-1)
    choice = torch.multinomial(probs, 1)
    return idx.gather(-1, choice).squeeze(-1)


@torch.no_grad()
def generate(model, prompts, tokenizer, max_new_tokens=384, target_words=128,
             temperature=0.7, top_k=50, top_p=0.92, check_every=8, suppress_eos=True):
    """
    prompts: list of token-id lists (any lengths). Returns a list of generated strings
    (the continuation only, trimmed to `target_words` words).
    Prompts are left-padded to a common length; padded keys are masked out, and because RoPE
    is relative, the shift does not change what the real tokens see.
    """
    model.eval()
    dev = next(model.parameters()).device
    B = len(prompts)
    L = max(len(p) for p in prompts)
    assert L + max_new_tokens <= model.cfg.max_seq_len, "prompt + continuation exceeds context"
    pad, eos = tokenizer.pad_id, tokenizer.eos_id

    ids = torch.full((B, L), pad, dtype=torch.long)
    valid = torch.zeros((B, L), dtype=torch.bool)
    for r, p in enumerate(prompts):
        ids[r, L - len(p):] = torch.tensor(p, dtype=torch.long)
        valid[r, L - len(p):] = True
    ids, valid = ids.to(dev), valid.to(dev)

    # prefill: causal mask AND key-validity; diagonal always allowed so pad rows never go all -inf
    causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dev))
    mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(L, dtype=torch.bool, device=dev)[None, None]
    logits, caches = model(ids, caches=None, start_pos=0, key_mask=mask, last_only=True)

    out = torch.empty((B, 0), dtype=torch.long, device=dev)
    finished = torch.zeros(B, dtype=torch.bool, device=dev)
    for step in range(max_new_tokens):
        step_logits = logits[:, -1].float()
        step_logits[:, pad] = float("-inf")
        if suppress_eos:                          # the task wants a full-length continuation: stop on word count only
            step_logits[:, eos] = float("-inf")
        nxt = _sample(step_logits, temperature, top_k, top_p)
        nxt = torch.where(finished, torch.full_like(nxt, pad), nxt)
        out = torch.cat([out, nxt[:, None]], dim=1)
        finished |= nxt == eos

        if (step + 1) % check_every == 0:
            for r in torch.nonzero(~finished).flatten().tolist():
                if len(tokenizer.decode(out[r].tolist()).split()) >= target_words:
                    finished[r] = True
        if bool(finished.all()):
            break

        valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=dev)], dim=1)
        step_mask = valid[:, None, None, :]
        logits, caches = model(nxt[:, None], caches=caches, start_pos=L + step,
                               key_mask=step_mask, last_only=True)

    texts = []
    for r in range(B):
        toks = out[r].tolist()
        if eos in toks:
            toks = toks[:toks.index(eos)]
        toks = [t for t in toks if t != pad]
        texts.append(" ".join(tokenizer.decode(toks).split()[:target_words]))
    return texts

Writing /kaggle/working/phase7_arch3/phase1_model.py


In [5]:
%%writefile /kaggle/working/phase7_arch3/hub_files.py
"""Finds Phase-1 files: an attached Kaggle dataset first, then a local cache, then Hugging Face."""
import os
import re

from huggingface_hub import hf_hub_download

KAGGLE_INPUT = "/kaggle/input"


def _squash(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())


def find_in_kaggle_input(filename, hint):
    """Only accepts a file whose path mentions the architecture (e.g. 'arch3', 'arch-3', 'Arch_3'),
    because arch 2, 3 and 4 all call their weights model_final.pt."""
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    hits = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            path = os.path.join(root, filename)
            if _squash(hint) in _squash(path):
                hits.append(path)
    return sorted(hits)[0] if hits else None


def fetch_from_hub(repo, filename, local_dir):
    try:
        return hf_hub_download(repo, filename, local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def locate(filenames, repos, hint, local_dir):
    """Names are tried in order of preference. Returns (path, where it came from)."""
    for name in filenames:
        path = find_in_kaggle_input(os.path.basename(name), hint)
        if path:
            return path, "kaggle input"
    for name in filenames:
        for repo in repos:
            cached = os.path.join(local_dir, _squash(repo), name)
            if os.path.exists(cached):
                return cached, f"cache of hf://{repo}"
            path = fetch_from_hub(repo, name, os.path.join(local_dir, _squash(repo)))
            if path:
                return path, f"hf://{repo}"
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input (path must mention '{hint}') or in "
                            f"{repos}. Attach the Phase-1 weights as a Kaggle dataset, or check HF_TOKEN and the repo names.")

Writing /kaggle/working/phase7_arch3/hub_files.py


In [6]:
%%writefile /kaggle/working/phase7_arch3/backbone.py
"""Loads the Phase-1 arch 3 weights (Llama-style decoder) and exposes them as a sentence encoder for rhetorical roles."""
import os

import torch
import torch.nn as nn

import config as C
from hub_files import locate, fetch_from_hub  # noqa: F401  (fetch_from_hub is used by run_rr)
from phase1_model import GPT, GPTConfig
from tokenizer import NerTokenizer


def add_prompt(x, mask, prompt):
    """Prompt tuning (Lester et al. 2021): prepend learned vectors to the token embeddings."""
    if prompt is None:
        return x, mask, 0
    P = prompt.size(0)
    x = torch.cat([prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], dim=1)
    mask = torch.cat([mask.new_ones(mask.size(0), P), mask], dim=1)
    return x, mask, P


def load_tokenizer():
    path, _ = locate(C.TOKENIZER_FILES, C.TOKENIZER_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    tok = NerTokenizer.load(path)
    tok.sep_id = tok.eos_id               # marks the start of every sentence inside a pack
    return tok


class Backbone(nn.Module):
    def __init__(self, gpt, keep_layers, bidirectional):
        super().__init__()
        self.d_model = gpt.cfg.d_model
        self.tok_emb = gpt.tok_emb
        blocks = gpt.blocks if keep_layers is None else [gpt.blocks[i] for i in keep_layers]
        self.layers = nn.ModuleList(blocks)
        self.final_norm = gpt.final_norm
        self.register_buffer("rope_cos", gpt.rope_cos.clone(), persistent=False)
        self.register_buffer("rope_sin", gpt.rope_sin.clone(), persistent=False)
        self.bidirectional = bidirectional
        for blk in self.layers:
            blk.attn.causal = not bidirectional

    def forward(self, ids, mask, prompt=None):
        x, mask, P = add_prompt(self.tok_emb(ids), mask, prompt)
        T = x.size(1)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = mask[:, None, None, :] if (self.bidirectional and not bool(mask.all())) else None
        for blk in self.layers:
            x, _ = blk(x, cos, sin, None, key_mask)
        return self.final_norm(x)[:, P:]


def build_backbone(keep_layers=None, bidirectional=True):
    path, source = locate(C.WEIGHT_FILES, C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    blob = torch.load(path, map_location="cpu")
    gpt = GPT(GPTConfig(**blob["model_cfg"]))
    gpt.load_state_dict({k: v.float() for k, v in blob["model"].items()})
    if os.environ.get("P2_VERBOSE", "1") == "1":
        print(f"Phase-1 arch 3 weights: {path} ({source})", flush=True)
    return Backbone(gpt, keep_layers, bidirectional)


def student_layers():
    return list(C.STUDENT_LAYERS)

Writing /kaggle/working/phase7_arch3/backbone.py


In [7]:
%%writefile /kaggle/working/phase7_arch3/pcr_core.py
"""
Phase 7 (prior case retrieval) machinery shared by all four architectures.

Ranking = dense similarity from the fine-tuned Phase-1 model, fused with BM25. The dense part is a bi-encoder: a case is
read as a few 512-token chunks spread over the whole judgment, the backbone encodes each chunk, chunks are mean-pooled
into one normalised vector, and queries and candidates share the encoder. It is trained contrastively (InfoNCE) with
in-batch negatives plus BM25 hard negatives. Every query also appears in its own candidate pool, so it is never
allowed to retrieve itself.
"""
import os
import re
import json
import math
from collections import Counter
from multiprocessing import Pool

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

TOP_K = 7


# ------------------------------------------------------------------------------------------- data
def load_split(query_path, cand_path):
    with open(query_path, encoding="utf-8") as f:
        queries = json.load(f)
    with open(cand_path, encoding="utf-8") as f:
        cands = json.load(f)
    as_text = lambda v: " ".join(v["text"]) if isinstance(v["text"], list) else v["text"]
    q_ids = list(queries)
    q_text = [as_text(queries[k]) for k in q_ids]
    rel = [list(queries[k].get("relevant_candidates", [])) for k in q_ids]
    c_ids = list(cands)
    c_text = [as_text(cands[k]) for k in c_ids]
    return q_ids, q_text, rel, c_ids, c_text


# ------------------------------------------------------------------------------------------ BM25
WORD = re.compile(r"[a-z0-9]+")


def words(text):
    return WORD.findall(text.lower())


class BM25:
    """Okapi BM25 over the candidate pool, with an inverted index so scoring touches only matching documents."""

    def __init__(self, docs, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.n = len(docs)
        self.lens = np.array([len(d) for d in docs], dtype=np.float32)
        self.avg = float(self.lens.mean())
        self.post = {}
        for i, d in enumerate(docs):
            for w, f in Counter(d).items():
                self.post.setdefault(w, []).append((i, f))
        self.idf = {w: math.log(1 + (self.n - len(p) + 0.5) / (len(p) + 0.5)) for w, p in self.post.items()}
        self.post = {w: (np.array([i for i, _ in p]), np.array([f for _, f in p], dtype=np.float32)) for w, p in self.post.items()}

    def scores(self, query_words, min_idf, max_terms):
        qc = Counter(w for w in query_words if self.idf.get(w, 0.0) > min_idf)
        terms = sorted(qc, key=lambda w: -qc[w] * self.idf[w])[:max_terms]      # the query's most telling words
        s = np.zeros(self.n, dtype=np.float32)
        norm = self.k1 * (1 - self.b + self.b * self.lens / self.avg)
        for w in terms:
            idx, f = self.post[w]
            s[idx] += self.idf[w] * f * (self.k1 + 1) / (f + norm[idx])
        return s


def bm25_matrix(q_text, c_text, cfg):
    bm = BM25([words(t) for t in c_text])
    return np.stack([bm.scores(words(t), cfg.BM25_MIN_IDF, cfg.BM25_MAX_TERMS) for t in q_text])


# ----------------------------------------------------------------------------------------- chunks
_tok = None
_cfg = None


def _init_worker(cfg_dict):
    global _tok, _cfg
    import backbone as BB
    _tok = BB.load_tokenizer()
    _cfg = cfg_dict


def chunk_text(text):
    """Up to K chunks of W tokens spread evenly over the case (first and last included)."""
    W, K = _cfg["W"], _cfg["K"]
    ids = _tok.encode(text[:_cfg["char_cap"]]) or [_cfg["pad"]]
    n = len(ids)
    starts = list(range(0, n, W)) if n <= K * W else [round(i * (n - W) / (K - 1)) for i in range(K)]
    out = np.full((K, W), _cfg["pad"], dtype=np.int32)
    lens = np.zeros(K, dtype=np.int16)
    for k, s in enumerate(starts[:K]):
        piece = ids[s:s + W]
        out[k, :len(piece)] = piece
        lens[k] = len(piece)
    return out, lens


def prepare_texts(texts, cfg_dict, workers):
    with Pool(workers, initializer=_init_worker, initargs=(cfg_dict,)) as pool:
        res = pool.map(chunk_text, texts, chunksize=16)
    return np.stack([r[0] for r in res]), np.stack([r[1] for r in res])


# ----------------------------------------------------------------------------------- adaptation
class LoRMALinear(nn.Module):
    """Bihany, Patel & Modi 2025 (LoRMA+): h = (alpha/r B A + I) W0 x, computed as W0x -> A -> B."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale
        return h + self.base.bias if self.base.bias is not None else h


class IA3Out(nn.Module):
    """Liu et al. 2022 (IA3): learned rescaling of a projection's output (keys / values). Starts at 1."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.scale


class IA3In(nn.Module):
    """IA3 on the feed-forward block: rescale the hidden activation just before the down projection."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.scale)


def wrap_linears(root, names, make, skip=("router",)):
    n = 0
    for parent in list(root.modules()):
        for name, child in list(parent.named_children()):
            if name in skip:
                continue
            if name in names and isinstance(child, nn.Linear):
                setattr(parent, name, make(child))
                n += 1
    return n


# ----------------------------------------------------------------------------------------- model
class Retriever(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone = backbone
        self.proj = nn.Linear(backbone.d_model, cfg.EMB_DIM)
        self.prompt = None

    def forward(self, ids, lens):
        """ids (B, K, W) -> L2-normalised case embeddings (B, EMB_DIM). Empty chunks never reach the backbone."""
        B, K, W = ids.shape
        flat_ids, flat_lens = ids.view(B * K, W), lens.view(B * K)
        real = flat_lens > 0
        L = int(flat_lens[real].max())
        mask = torch.arange(L, device=ids.device)[None, :] < flat_lens[real][:, None]
        h = self.backbone(flat_ids[real][:, :L], mask, prompt=self.prompt).float()
        pooled = (h * mask[..., None]).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        chunks = torch.zeros(B * K, pooled.size(-1), device=ids.device).index_put((real.nonzero().squeeze(1),), pooled)
        present = real.view(B, K, 1).float()
        doc = (chunks.view(B, K, -1) * present).sum(1) / present.sum(1).clamp(min=1)
        return F.normalize(self.proj(doc), dim=-1)


def configure_method(model, method, cfg):
    bb = model.backbone
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
        return "all weights trainable"
    for p in bb.parameters():
        p.requires_grad = False
    if method == "lorma":
        n = wrap_linears(bb.layers, cfg.ADAPTER_TARGETS, lambda m: LoRMALinear(m, cfg.ADAPTER_RANK, cfg.LORMA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRMA on {n} attention projections (r={cfg.ADAPTER_RANK})"
    if method == "ia3":
        n_out = wrap_linears(bb.layers, cfg.IA3_OUT_TARGETS, IA3Out)
        n_in = wrap_linears(bb.layers, cfg.IA3_IN_TARGETS, IA3In)
        return f"IA3 vectors on {n_out} key/value projections and {n_in} feed-forward layers"
    if method == "partial":
        for layer in list(bb.layers)[-cfg.PARTIAL_TOP_LAYERS:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in bb.final_norm.parameters():
            p.requires_grad = True
        return f"top {cfg.PARTIAL_TOP_LAYERS} backbone layers + final norm trainable"
    if method == "prompt":
        # Lester et al. 2021: learned virtual tokens in front of every chunk, initialised from real token embeddings
        g = torch.Generator().manual_seed(cfg.SEED)
        w = bb.tok_emb.weight
        pick = torch.randint(100, w.size(0) - 100, (cfg.PROMPT_LEN,), generator=g)
        model.prompt = nn.Parameter(w[pick].detach().clone())
        return f"{cfg.PROMPT_LEN} soft-prompt vectors in front of every chunk"
    raise ValueError(method)


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().float().half() for k, v in model.state_dict().items() if k in names}


def build_optimizer(model, method, cfg):
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name == "prompt":
            lr = cfg.PROMPT_LR
        elif not name.startswith("backbone."):
            lr = cfg.HEAD_LR
        elif method in ("full", "kd"):
            lr = cfg.FULL_LR
        elif method == "partial":
            lr = cfg.PARTIAL_LR
        else:
            lr = cfg.ADAPTER_LR
        wd = 0.0 if (p.ndim < 2 or name == "prompt") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)


# --------------------------------------------------------------------------------------- scoring
@torch.no_grad()
def embed(model, ids, lens, device, batch):
    model.eval()
    out = []
    for b in range(0, len(ids), batch):
        x = torch.from_numpy(ids[b:b + batch].astype(np.int64)).to(device)
        l = torch.from_numpy(lens[b:b + batch].astype(np.int64)).to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            out.append(model(x, l).float().cpu())
    return torch.cat(out).numpy()


def zscore(m):
    return (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-6)


def rank(scores, q_ids, c_ids):
    """Top-7 candidate ids per query; a query's own id is never returned."""
    pos = {c: j for j, c in enumerate(c_ids)}
    s = scores.copy()
    out = []
    for i, q in enumerate(q_ids):
        if q in pos:
            s[i, pos[q]] = -np.inf
        top = np.argsort(-s[i])[:TOP_K]
        out.append([c_ids[j] for j in top])
    return out


def micro_f1_at_7(preds, rel):
    tp = sum(len(set(p) & set(r)) for p, r in zip(preds, rel))
    n_pred = sum(len(p) for p in preds)
    n_gold = sum(len(r) for r in rel)
    p = tp / max(1, n_pred)
    r = tp / max(1, n_gold)
    return {"f1": 2 * p * r / (p + r) if p + r else 0.0, "p": p, "r": r}


def tune_fusion(dense, bm25, q_ids, c_ids, rel, grid):
    """Weight alpha on the dense score (1 - alpha on BM25) that gives the best dev micro-F1@7."""
    dz, bz = zscore(dense), zscore(bm25)
    results = []
    for a in grid:
        results.append((a, micro_f1_at_7(rank(a * dz + (1 - a) * bz, q_ids, c_ids), rel)))
    best = max(results, key=lambda t: t[1]["f1"])
    return best[0], results


def write_submission(q_ids, preds, out_dir, zip_name, pool):
    import zipfile
    pool = set(pool)
    payload = {}
    for q, p in zip(q_ids, preds):
        assert len(p) == TOP_K == len(set(p)) and all(c in pool and c != q for c in p), q
        payload[q] = p
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w") as f:
        json.dump(payload, f)
    zip_path = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
    return path, zip_path

Writing /kaggle/working/phase7_arch3/pcr_core.py


In [8]:
%%writefile /kaggle/working/phase7_arch3/run_pcr.py
"""
One Phase-7 experiment = one call:

    python run_pcr.py --method lorma

Contrastive training of the bi-encoder for a fixed wall-clock budget (best checkpoint by dev micro-F1@7), then the dense
scores are fused with BM25 - the mixing weight is tuned on dev - and the top 7 test candidates are written out.
"""
import os
import sys
import json
import math
import time
import random
import argparse

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
import backbone as BB
from hub_files import fetch_from_hub
from pcr_core import (Retriever, configure_method, count_params, trainable_state, build_optimizer, embed, rank,
                      micro_f1_at_7, tune_fusion, write_submission, zscore)


def log(*a):
    print(*a, flush=True)


def build_model(method, keep_layers=None):
    model = Retriever(BB.build_backbone(keep_layers=keep_layers, bidirectional=C.BIDIRECTIONAL), C)
    note = configure_method(model, method, C)
    return model, note


def load_finetuned(method, device):
    path = os.path.join(C.OUT_DIR, method, f"pcr_{method}.pt")
    if not os.path.exists(path):
        path = fetch_from_hub(C.PCR_REPO, f"{method}/pcr_{method}.pt", os.path.join(C.OUT_DIR, "_hub"))
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"No fine-tuned '{method}' model locally or on {C.PCR_REPO}; run '{method}' first.")
    blob = torch.load(path, map_location="cpu")
    model, _ = build_model(method, blob.get("keep_layers"))
    _, unexpected = model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    assert not unexpected, unexpected
    return model.to(device).eval()


def push(files, message):
    if not os.environ.get("HF_TOKEN"):
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(C.PCR_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
        for local, remote in files:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.PCR_REPO,
                            token=os.environ["HF_TOKEN"], commit_message=message)
        log(f"pushed to {C.PCR_REPO}: {message}")
    except Exception as e:
        log(f"HF push failed ({type(e).__name__}: {e}) - the files are still in {C.OUT_DIR}")


def load(split):
    z = np.load(os.path.join(C.DATA_DIR, f"{split}.npz"))
    meta = json.load(open(os.path.join(C.DATA_DIR, f"{split}_meta.json")))
    return {k: z[k] for k in z.files}, meta


def dense_scores(model, d, device):
    q = embed(model, d["q_ids"], d["q_lens"], device, C.EVAL_BATCH)
    c = embed(model, d["c_ids"], d["c_lens"], device, C.EVAL_BATCH)
    return q @ c.T


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["full", "lorma", "ia3", "partial", "prompt", "kd"])
    method = ap.parse_args().method
    random.seed(C.SEED)
    np.random.seed(C.SEED)
    torch.manual_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = os.path.join(C.OUT_DIR, method)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    tr, tr_meta = load("train")
    dv, dv_meta = load("dev")
    te, te_meta = load("test")
    c_index = {c: j for j, c in enumerate(tr_meta["c_ids"])}
    # (query row, positive candidate rows, hard-negative candidate rows) for every training query with a positive
    examples = []
    for i, (q, rel) in enumerate(zip(tr_meta["q_ids"], tr_meta["rel"])):
        pos = [c_index[c] for c in rel if c in c_index and c != q]
        if pos:
            examples.append((i, pos, [j for j in tr["hard_neg"][i] if j >= 0], c_index.get(q, -1)))
    if C.TRAIN_QUERY_LIMIT:
        examples = examples[:C.TRAIN_QUERY_LIMIT]
    log(f"[{method}] train queries {len(examples)} (pool {len(tr_meta['c_ids'])}) | dev {len(dv_meta['q_ids'])} "
        f"(pool {len(dv_meta['c_ids'])}) | test {len(te_meta['q_ids'])} (pool {len(te_meta['c_ids'])})")

    keep = BB.student_layers() if method == "kd" else None
    model, note = build_model(method, keep)
    model.to(device)
    total, trainable = count_params(model)
    log(f"[{method}] {note} | params {total / 1e6:.1f}M, trainable {trainable / 1e6:.2f}M | budget {C.TRAIN_MINUTES} min")

    teacher = None
    if method == "kd":
        teacher = load_finetuned(C.KD_TEACHER, device)
        log(f"[kd] teacher = fine-tuned '{C.KD_TEACHER}' ({count_params(teacher)[0] / 1e6:.1f}M) -> student keeping "
            f"layers {keep} ({total / 1e6:.1f}M); it learns the teacher's similarity distribution and embeddings")

    optimizer = build_optimizer(model, method, C)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    budget = C.TRAIN_MINUTES * 60
    step_cap = max(1, C.MAX_EPOCHS * len(examples) // (C.MICRO_QUERIES * C.GRAD_ACCUM))
    rng = np.random.RandomState(C.SEED)
    order = rng.permutation(len(examples))
    cursor, step, seen, running = 0, 0, 0, 0.0
    best, history = {"f1": -1.0, "step": 0}, []
    t_train = time.time()

    def progress():
        return max(step / step_cap, (time.time() - t_train) / budget)

    def as_tensor(rows_ids, rows_lens):
        return (torch.from_numpy(rows_ids.astype(np.int64)).to(device), torch.from_numpy(rows_lens.astype(np.int64)).to(device))

    def sample_chunks(ids, lens):
        # training reads TRAIN_CHUNKS random chunks of each case (all of them at evaluation) - cheaper and a mild augmentation
        k = C.TRAIN_CHUNKS
        if k >= ids.shape[1]:
            return ids, lens
        out_ids = np.zeros((len(ids), k, ids.shape[2]), dtype=ids.dtype)
        out_lens = np.zeros((len(ids), k), dtype=lens.dtype)
        for r in range(len(ids)):
            real = np.flatnonzero(lens[r] > 0)
            pick = np.sort(rng.choice(real, size=min(k, len(real)), replace=False))
            out_ids[r, :len(pick)], out_lens[r, :len(pick)] = ids[r, pick], lens[r, pick]
        return out_ids, out_lens

    while progress() < 1.0:
        warm = min(1.0, (step + 1) / C.WARMUP_STEPS)
        lr_scale = warm * (C.MIN_LR_RATIO + (1 - C.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, progress()))))
        for g, lr in zip(optimizer.param_groups, base_lrs):
            g["lr"] = lr * lr_scale
        model.train()
        for _ in range(C.GRAD_ACCUM):
            if cursor + C.MICRO_QUERIES > len(order):
                order, cursor = rng.permutation(len(examples)), 0
            batch = [examples[k] for k in order[cursor:cursor + C.MICRO_QUERIES]]
            cursor += C.MICRO_QUERIES
            q_rows = [b[0] for b in batch]
            pos_rows = [b[1][rng.randint(len(b[1]))] for b in batch]
            neg_rows = [b[2][rng.randint(len(b[2]))] if b[2] else rng.randint(len(tr["c_ids"])) for b in batch]
            cand_rows = pos_rows + neg_rows
            # a candidate that is relevant to another query in the batch must not count as its negative
            relevant = [set(b[1]) | {b[3]} for b in batch]                 # ... and neither may its own copy in the pool
            mask = torch.tensor([[j != i and cand_rows[j] in relevant[i] for j in range(len(cand_rows))]
                                 for i in range(len(batch))], device=device)
            target = torch.arange(len(batch), device=device)
            q_in = as_tensor(*sample_chunks(tr["q_ids"][q_rows], tr["q_lens"][q_rows]))
            c_in = as_tensor(*sample_chunks(tr["c_ids"][cand_rows], tr["c_lens"][cand_rows]))
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                q_emb = model(*q_in)
                c_emb = model(*c_in)
                logits = (q_emb.float() @ c_emb.float().T / C.TEMPERATURE).masked_fill(mask, -1e4)
                loss = F.cross_entropy(logits, target)                              # InfoNCE
                if teacher is not None:
                    with torch.no_grad():
                        tq = teacher(*q_in).float()
                        tc = teacher(*c_in).float()
                        t_logits = (tq @ tc.T / C.TEMPERATURE).masked_fill(mask, -1e4)
                    T = C.KD_TEMPERATURE
                    kl = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(t_logits / T, -1), reduction="batchmean") * T * T
                    rep = 2 - F.cosine_similarity(q_emb.float(), tq).mean() - F.cosine_similarity(c_emb.float(), tc).mean()
                    loss = C.KD_ALPHA * loss + (1 - C.KD_ALPHA) * kl + C.KD_HIDDEN_WEIGHT * rep
            scaler.scale(loss / C.GRAD_ACCUM).backward()
            running += loss.item()
            seen += 1
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % C.EVAL_EVERY_STEPS == 0 or progress() >= 1.0:
            dense = dense_scores(model, dv, device)
            alpha, _ = tune_fusion(dense, dv["bm25"], dv_meta["q_ids"], dv_meta["c_ids"], dv_meta["rel"], C.FUSION_GRID)
            hybrid = micro_f1_at_7(rank(alpha * zscore(dense) + (1 - alpha) * zscore(dv["bm25"]), dv_meta["q_ids"], dv_meta["c_ids"]), dv_meta["rel"])
            dense_only = micro_f1_at_7(rank(dense, dv_meta["q_ids"], dv_meta["c_ids"]), dv_meta["rel"])
            history.append({"step": step, "train_loss": running / max(1, seen), "dev_dense_f1": dense_only["f1"],
                            "dev_hybrid_f1": hybrid["f1"], "alpha": alpha, "minutes": (time.time() - t0) / 60})
            log(f"[{method}] step {step} | loss {running / max(1, seen):.4f} | dev F1@7 dense {dense_only['f1']:.4f}, "
                f"hybrid {hybrid['f1']:.4f} (alpha {alpha:.1f}) | {100 * min(1.0, progress()):.0f}% of budget")
            running, seen = 0.0, 0
            if hybrid["f1"] > best["f1"]:
                best = {"f1": hybrid["f1"], "step": step}
                torch.save({"method": method, "keep_layers": keep, "weights": trainable_state(model), "step": step},
                           os.path.join(out_dir, f"pcr_{method}.pt"))

    # --------------------------------------- best checkpoint: fusion weight on dev, then rank the test pool
    blob = torch.load(os.path.join(out_dir, f"pcr_{method}.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    t_inf = time.time()
    dense = dense_scores(model, dv, device)
    dev_seconds = time.time() - t_inf
    alpha, grid = tune_fusion(dense, dv["bm25"], dv_meta["q_ids"], dv_meta["c_ids"], dv_meta["rel"], C.FUSION_GRID)
    scores = {a: r for a, r in grid}
    dense_only = micro_f1_at_7(rank(dense, dv_meta["q_ids"], dv_meta["c_ids"]), dv_meta["rel"])
    bm25_only = micro_f1_at_7(rank(dv["bm25"], dv_meta["q_ids"], dv_meta["c_ids"]), dv_meta["rel"])

    test_dense = dense_scores(model, te, device)
    preds = rank(alpha * zscore(test_dense) + (1 - alpha) * zscore(te["bm25"]), te_meta["q_ids"], te_meta["c_ids"])
    _, zip_path = write_submission(te_meta["q_ids"], preds, out_dir, f"submission_{C.ARCH}_{method}.zip", te_meta["c_ids"])

    summary = {
        "arch": C.ARCH, "method": method, "best_step": best["step"], "alpha": alpha,
        "dev_f1": scores[alpha]["f1"], "dev_p": scores[alpha]["p"], "dev_r": scores[alpha]["r"],
        "dev_dense_only_f1": dense_only["f1"], "dev_bm25_only_f1": bm25_only["f1"],
        "fusion_grid": [{"alpha": a, "f1": r["f1"]} for a, r in grid],
        "params_total": total, "params_trainable": trainable, "dev_encode_seconds": round(dev_seconds, 1),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else 0.0,
        "minutes": round((time.time() - t0) / 60, 1), "steps": step, "history": history,
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    log(f"[{method}] DONE | dev micro-F1@7 {scores[alpha]['f1']:.4f} with alpha {alpha:.1f} (dense only {dense_only['f1']:.4f}, "
        f"BM25 only {bm25_only['f1']:.4f}) -> {zip_path}")

    push([(os.path.join(out_dir, f"pcr_{method}.pt"), f"{method}/pcr_{method}.pt"),
          (os.path.join(out_dir, "results.json"), f"{method}/results.json"),
          (zip_path, f"{method}/{os.path.basename(zip_path)}")], f"{C.ARCH} {method}: dev F1@7 {scores[alpha]['f1']:.4f}")



if __name__ == "__main__":
    main()

Writing /kaggle/working/phase7_arch3/run_pcr.py


## 4. Data: download, tokenize, BM25
Done once with all CPU cores. Each split is stored as token windows for queries and pool, plus the BM25 scores
(dev / test) or the BM25 hard negatives (train). The BM25-only dev score printed here is the baseline to beat.

In [9]:
import importlib
import config as C
importlib.reload(C)
import backbone as BB
import pcr_core as R
importlib.reload(R)

for split, folder in C.SPLIT_FOLDERS.items():
    for name in (f"{split}.json", f"{split}_candidates.json"):
        hf_hub_download(C.HF_DATASET_REPO, f"Phase_7_PCR/{folder}/{name}", repo_type="dataset",
                        local_dir=C.DATA_DIR, token=HF_TOKEN)

tok = BB.load_tokenizer()                       # caches the tokenizer before the worker processes start
chunk_cfg = {"W": C.CHUNK_TOKENS, "K": C.MAX_CHUNKS, "char_cap": C.CHAR_CAP, "pad": tok.pad_id}

for split in ("train", "dev", "test"):
    t0 = time.time()
    q_ids, q_text, rel, c_ids, c_text = R.load_split(*C.RAW[split])
    qi, ql = R.prepare_texts(q_text, chunk_cfg, C.PREP_WORKERS)
    ci, cl = R.prepare_texts(c_text, chunk_cfg, C.PREP_WORKERS)
    bm25 = R.bm25_matrix(q_text, c_text, C)
    arrays = {"q_ids": qi, "q_lens": ql, "c_ids": ci, "c_lens": cl}
    pos = {c: j for j, c in enumerate(c_ids)}
    extra = ""
    if split == "train":
        hard = np.full((len(q_ids), C.HARD_NEG_POOL), -1, dtype=np.int32)
        for i, q in enumerate(q_ids):
            banned = {pos[c] for c in rel[i] if c in pos} | ({pos[q]} if q in pos else set())
            top = [j for j in np.argsort(-bm25[i])[:C.HARD_NEG_POOL + len(banned)] if j not in banned][:C.HARD_NEG_POOL]
            hard[i, :len(top)] = top
        arrays["hard_neg"] = hard
    else:
        arrays["bm25"] = bm25.astype(np.float32)
    if split != "test" and any(rel):
        extra = f" | relevant per query {np.mean([len(r) for r in rel]):.1f} | BM25-only micro-F1@7 " \
                f"{R.micro_f1_at_7(R.rank(bm25, q_ids, c_ids), rel)['f1']:.4f}"
    np.savez(os.path.join(C.DATA_DIR, f"{split}.npz"), **arrays)
    json.dump({"q_ids": q_ids, "c_ids": c_ids, "rel": rel}, open(os.path.join(C.DATA_DIR, f"{split}_meta.json"), "w"))
    print(f"{split}: {len(q_ids)} queries, pool {len(c_ids):,} | windows per case {(cl > 0).sum(1).mean():.2f} of "
          f"{C.MAX_CHUNKS} | {time.time() - t0:.0f}s{extra}")
    del q_text, c_text, arrays, bm25
    gc.collect()

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

... (2 outputs left out)

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/413k [00:00<?, ?B/s]

train: 827 queries, pool 4,320 | windows per case 2.89 of 3 | 33s | relevant per query 6.4 | BM25-only micro-F1@7 0.1889
dev: 118 queries, pool 1,023 | windows per case 2.89 of 3 | 9s | relevant per query 8.2 | BM25-only micro-F1@7 0.2412
test: 237 queries, pool 1,727 | windows per case 2.90 of 3 | 13s


## 5. Load the Phase-1 weights once
This checks the encoder end to end and caches the files for the experiment processes.

In [10]:
probe = R.Retriever(BB.build_backbone(bidirectional=C.BIDIRECTIONAL), C)
dev = np.load(os.path.join(C.DATA_DIR, "dev.npz"))
with torch.no_grad():
    e = probe(torch.from_numpy(dev["q_ids"][:2].astype(np.int64)), torch.from_numpy(dev["q_lens"][:2].astype(np.int64)))
print(f"model ready: {sum(p.numel() for p in probe.parameters()) / 1e6:.1f}M parameters | embeddings {tuple(e.shape)}, "
      f"norms {e.norm(dim=-1).tolist()}")
del probe, dev, e

model_final.pt: reconstructing file:   0%|          |  0.00B /  200MB            

model_final.pt: downloading bytes:           |  0.00B            

Phase-1 arch 3 weights: /kaggle/working/phase7_arch3/phase1/d240357legalarch3decoder/model_final.pt (hf://D-240357/legal-arch3-decoder)
model ready: 100.3M parameters | embeddings (2, 256), norms [1.0, 1.0]


## 6. Run the experiments
Each experiment is its own process (`run_pcr.py --method ...`); with two GPUs, two run at once, so the six take about
three rounds of `TRAIN_MINUTES` + a few minutes. `kd` starts as soon as `full` (its teacher) is done.
Results, checkpoints and zips are pushed to `D-240357/legal-pcr-arch3`.

In [11]:
import subprocess, threading

pending = list(C.METHODS)
done, failed = set(), set()
lock = threading.Lock()


def next_job():
    with lock:
        if "kd" in pending and C.KD_TEACHER in failed:
            pending.remove("kd")
            print(f"skipping kd: its teacher '{C.KD_TEACHER}' failed")
        for m in pending:
            if m != "kd" or C.KD_TEACHER in done or C.KD_TEACHER not in C.METHODS:
                pending.remove(m)
                return m
        return "wait" if pending else None


def worker(gpu):
    while True:
        job = next_job()
        if job is None:
            return
        if job == "wait":
            time.sleep(15)
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        proc = subprocess.Popen([sys.executable, "run_pcr.py", "--method", job], cwd=WORK_DIR, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="")
        with lock:
            (done if proc.wait() == 0 else failed).add(job)


t0 = time.time()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nfinished in {(time.time() - t0) / 60:.0f} min | ok: {sorted(done)} | failed: {sorted(failed)}")

[gpu1] [lorma] train queries 817 (pool 4320) | dev 118 (pool 1023) | test 237 (pool 1727)
[gpu0] [full] train queries 817 (pool 4320) | dev 118 (pool 1023) | test 237 (pool 1727)
[gpu1] Phase-1 arch 3 weights: /kaggle/working/phase7_arch3/phase1/d240357legalarch3decoder/model_final.pt (cache of hf://D-240357/legal-arch3-decoder)
[gpu0] Phase-1 arch 3 weights: /kaggle/working/phase7_arch3/phase1/d240357legalarch3decoder/model_final.pt (cache of hf://D-240357/legal-arch3-decoder)
[gpu0] [full] all weights trainable | params 100.3M, trainable 100.29M | budget 30 min
[gpu1] [lorma] LoRMA on 24 attention projections (r=8) | params 100.7M, trainable 0.59M | budget 30 min
[gpu1] [lorma] step 100 | loss 1.2890 | dev F1@7 dense 0.2524, hybrid 0.3224 (alpha 0.8) | 8% of budget
[gpu0] [full] step 100 | loss 1.2794 | dev F1@7 dense 0.2312, hybrid 0.3102 (alpha 0.9) | 9% of budget
[gpu1] [lorma] step 200 | loss 1.1886 | dev F1@7 dense 0.2513, hybrid 0.3235 (alpha 0.8) | 16% of budget
[gpu0] [full] 

## 7. Results
Dev micro-F1@7 of the fused ranking, next to dense-only and BM25-only, and the cost of each experiment.

In [12]:
import pandas as pd

rows = []
for path in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "results.json"))):
    r = json.load(open(path))
    rows.append({"method": r["method"], "dev F1@7": r["dev_f1"], "P": r["dev_p"], "R": r["dev_r"], "alpha": r["alpha"],
                 "dense only": r["dev_dense_only_f1"], "BM25 only": r["dev_bm25_only_f1"], "best step": r["best_step"],
                 "params (M)": r["params_total"] / 1e6, "trainable %": 100 * r["params_trainable"] / r["params_total"],
                 "dev encode s": r["dev_encode_seconds"], "minutes": r["minutes"], "peak GPU GB": r["peak_gpu_gb"]})
table = pd.DataFrame(rows).sort_values("dev F1@7", ascending=False).round(4)
display(table)
best = table.iloc[0]["method"]

,method,dev F1@7,P,R,alpha,dense only,BM25 only,best step,params (M),trainable %,dev encode s,minutes,peak GPU GB
3,lorma,0.3513,0.3826,0.3248,0.80,0.2779,0.2412,1000,100.6828,0.5861,30.7,32.0,5.44
4,partial,0.3413,0.3717,0.3155,0.85,0.2624,0.2412,500,100.2895,25.2964,27.9,21.7,2.79
1,ia3,0.3402,0.3705,0.3145,0.80,0.2646,0.2412,1100,100.3295,0.2360,31.7,32.0,5.61
0,full,0.3391,0.3692,0.3135,0.85,0.2601,0.2412,500,100.2895,100.0000,28.0,31.9,7.58
5,prompt,0.3380,0.3680,0.3124,0.80,0.2468,0.2412,900,100.3018,0.2085,29.4,30.7,4.69
2,kd,0.3313,0.3608,0.3063,0.80,0.2390,0.2412,1200,62.5316,100.0000,14.3,25.8,4.83


## 8. Pick a submission
Every experiment wrote `runs/<method>/submission_<arch>_<method>.zip`: a single `prediction.json` at the root mapping
each of the 237 test query ids to 7 distinct candidate ids from the test pool (never the query itself). The best on
dev is also copied to `/kaggle/working/submission.zip`. Upload the zips exactly as they are.

In [13]:
shutil.copy(glob.glob(os.path.join(C.OUT_DIR, best, "submission_*.zip"))[0], "/kaggle/working/submission.zip")
print(f"best on dev micro-F1@7: {best} -> /kaggle/working/submission.zip")
for z in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "submission_*.zip"))):
    print("  ", z)

best on dev micro-F1@7: lorma -> /kaggle/working/submission.zip
   /kaggle/working/phase7_arch3/runs/full/submission_arch3_full.zip
   /kaggle/working/phase7_arch3/runs/ia3/submission_arch3_ia3.zip
   /kaggle/working/phase7_arch3/runs/kd/submission_arch3_kd.zip
   /kaggle/working/phase7_arch3/runs/lorma/submission_arch3_lorma.zip
   /kaggle/working/phase7_arch3/runs/partial/submission_arch3_partial.zip
   /kaggle/working/phase7_arch3/runs/prompt/submission_arch3_prompt.zip
